In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import udf, col
from pyspark.ml.linalg import Vectors, VectorUDT
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator
import time




In [ ]:
# ==========================================
# 1. KHỞI TẠO SPARK VỚI CẤU HÌNH BIG DATA
# ==========================================
print(" Đang khởi tạo SparkSession...")
spark = SparkSession.builder \
    .appName("HM_Spark_KMeans_Clustering") \
    .config("spark.driver.memory", "8g") \
    .config("spark.executor.memory", "8g") \
    .config("spark.sql.execution.arrow.pyspark.enabled", "true") \
    .getOrCreate()

 Đang khởi tạo SparkSession...


In [ ]:
# ==========================================
# 2. ĐỌC DỮ LIỆU & CHUYỂN ĐỔI ĐỊNH DẠNG VECTOR
# ==========================================
print(" Đang đọc dữ liệu vector CLIP từ Parquet...")
vector_path = "/content/drive/MyDrive/Project Data/output_vectors.parquet"
df_vectors = spark.read.parquet(vector_path)

# Hàm UDF: Ép mảng Float 512 chiều thành định dạng DenseVector của Spark MLlib
print(" Đang ép kiểu sang Spark DenseVector...")
to_vector_udf = udf(lambda a: Vectors.dense(a), VectorUDT())

# Chỉ lấy dòng không bị null và tạo cột 'features' chuẩn MLlib
df_ml = df_vectors.dropna(subset=["clip_vector"]) \
                  .withColumn("features", to_vector_udf(col("clip_vector")))

# Lưu cache bảng này vào RAM phân tán để vòng lặp K-Means chạy nhanh hơn
df_ml.cache()

 Đang đọc dữ liệu vector CLIP từ Parquet...
 Đang ép kiểu sang Spark DenseVector...


DataFrame[article_id: string, clip_vector: array<float>, features: vector]

In [ ]:
# ==========================================
# 3. CHẠY VÒNG LẶP TÌM SỐ CỤM (K) TỐI ƯU
# ==========================================
print(" Đang tìm số lượng cụm (K) tối ưu bằng Silhouette Score...")

# Thử nghiệm các mức K khác nhau để xem phong cách thời trang chia làm mấy nhóm là đẹp
k_values = [10, 15, 20, 25, 30]
best_k = 0
best_score = -1
best_model = None

# Trình đánh giá chất lượng cụm
evaluator = ClusteringEvaluator(predictionCol="prediction", featuresCol="features",
                                metricName="silhouette", distanceMeasure="squaredEuclidean")

for k in k_values:
    start_time = time.time()

    # Cấu hình K-Means (seed để kết quả ổn định qua các lần chạy)
    kmeans = KMeans(k=k, seed=42, featuresCol="features", predictionCol="prediction", maxIter=20)

    # Huấn luyện mô hình phân tán
    model = kmeans.fit(df_ml)

    # Gắn nhãn và chấm điểm
    predictions = model.transform(df_ml)
    score = evaluator.evaluate(predictions)

    end_time = time.time()
    print(f"   -> Với K={k:2d} | Silhouette Score = {score:.4f} | Thời gian chạy: {end_time - start_time:.2f}s")

    # Lưu lại model có điểm cao nhất
    if score > best_score:
        best_score = score
        best_k = k
        best_model = model

print("-" * 50)
print(f" CHỐT MÔ HÌNH TỐT NHẤT: K = {best_k} (Điểm Silhouette = {best_score:.4f})")
print("-" * 50)

 Đang tìm số lượng cụm (K) tối ưu bằng Silhouette Score...
   -> Với K=10 | Silhouette Score = 0.0893 | Thời gian chạy: 194.07s
   -> Với K=15 | Silhouette Score = 0.1238 | Thời gian chạy: 141.92s
   -> Với K=20 | Silhouette Score = 0.0905 | Thời gian chạy: 148.43s
   -> Với K=25 | Silhouette Score = 0.0984 | Thời gian chạy: 153.04s
   -> Với K=30 | Silhouette Score = 0.0914 | Thời gian chạy: 160.75s
--------------------------------------------------
 CHỐT MÔ HÌNH TỐT NHẤT: K = 15 (Điểm Silhouette = 0.1238)
--------------------------------------------------


In [ ]:
# ==========================================
# 4. GẮN NHÃN BẰNG MÔ HÌNH TỐT NHẤT & LƯU KẾT QUẢ
# ==========================================
print(" Đang gắn nhãn phong cách cho toàn bộ 105.000 sản phẩm...")
final_predictions = best_model.transform(df_ml)

# Đổi tên cột prediction thành cluster_id cho dễ hiểu
df_final = final_predictions.withColumnRenamed("prediction", "cluster_id") \
                            .select("article_id", "cluster_id")

# Hiển thị thử 10 dòng
df_final.show(10)

# Lưu kết quả xuống ổ đĩa để team Dashboard lấy vẽ biểu đồ
output_path = "/content/drive/MyDrive/Project Data/article_clusters.parquet"
df_final.write.mode("overwrite").parquet(output_path)
print(f" Đã lưu kết quả phân cụm thành công tại: {output_path}")

# Giải phóng RAM
df_ml.unpersist()

 Đang gắn nhãn phong cách cho toàn bộ 105.000 sản phẩm...
+----------+----------+
|article_id|cluster_id|
+----------+----------+
|0647338014|         3|
|0696791001|         4|
|0740790005|        11|
|0749968003|         1|
|0848974001|         4|
|0897505002|         9|
|0176754019|         4|
|0478298002|        11|
|0478646003|         4|
|0553932003|         3|
+----------+----------+
only showing top 10 rows
 Đã lưu kết quả phân cụm thành công tại: /content/drive/MyDrive/Project Data/article_clusters.parquet


DataFrame[article_id: string, clip_vector: array<float>, features: vector]